# Group exercise: write the DSL correction

**Question:** are Facebook ads less negative than TV ads?

The setup is done for you: the ads, the true answer, LLM labels for every ad, and two baseline estimators. Your group writes **DSL**, then checks it against the truth.

Run the setup cells (Kernel → Restart & Run All is fine), then start at **Part 1**.

## Setup (run, don't edit)

In [ ]:
from dsl_demo import load_ads, llm_labels, ols, draw   # first: imports torch before scikit-learn
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold

MODEL     = "Qwen/Qwen2.5-0.5B-Instruct"   # small, fast; bigger models are not better with this prompt
MAX_CHARS = 600     # characters of ad text shown to the LLM
N_CORPUS  = 1000    # ads in our corpus
N_EXPERT  = 75      # ads with expert labels (~7.5%, like the paper)
REPEATS   = 200
SEED      = 6870

In [ ]:
ads = load_ads(N_CORPUS, SEED)
TRUTH, _ = ols(ads.Y, ads.X)
print(ads.groupby("platform").Y.mean().round(3))
print(f"True Facebook effect: {TRUTH:.3f}")

In [ ]:
SYSTEM = (
    "You code political ads. Is the primary purpose of the ad to promote a specific candidate, "
    "attack a candidate, or contrast the candidates?\n"
    "Promote: only talks about the sponsoring candidate.\n"
    "Contrast: talks about both the sponsor and the opponent.\n"
    "Attack: only talks about the opponent.\n"
    "Answer with one word: Promote, Contrast, or Attack."
)
ads["Q_tone"] = llm_labels(ads, MODEL, SYSTEM, MAX_CHARS)   # saved labels if we have them, else runs the LLM
ads["Q"] = ads.Q_tone.isin(["Attack", "Contrast"]).astype(int)
print("LLM agrees with experts:", round((ads.Q == ads.Y).mean(), 3))
print(ads.assign(agree=ads.Q == ads.Y).groupby("platform").agree.mean().round(3))

In [ ]:
PI = N_EXPERT / N_CORPUS               # every ad has the same, known chance of being expert-labelled
R = draw(N_CORPUS, N_EXPERT, 1)        # R[i] = 1 if the experts labelled ad i

def expert_labels(R):
    """What you would really have: the expert label Y where R == 1, and NaN (unknown) everywhere else."""
    return ads.Y.where(R == 1)

print(f"{R.sum()} of {N_CORPUS} ads have an expert label; PI = {PI:.3f}")
expert_labels(R).value_counts(dropna=False)   # 1 = negative, 0 = not, NaN = unknown

In [ ]:
def llm_only(R):                       # just trust the LLM labels
    return ols(ads.Q, ads.X)

T = TfidfVectorizer(min_df=2, max_features=5000).fit_transform(ads.text)   # word features, built once

def supervised(R):                     # train a text classifier on the expert labels, predict the rest
    lab = R == 1
    clf = LogisticRegression(max_iter=2000).fit(T[lab], ads.Y[lab])
    y = np.where(lab, ads.Y, clf.predict_proba(T)[:, 1])
    return ols(y, ads.X)

for name, f in [("LLM only", llm_only), ("Supervised", supervised)]:
    b, se = f(R)
    print(f"{name:11s} {b:6.3f}  [{b - 1.96*se:6.3f}, {b + 1.96*se:6.3f}]")
print(f"{'Truth':11s} {TRUTH:6.3f}")

Neither interval covers the truth. Why is each one off?

## What you have

| name | what it is |
|---|---|
| `ads.Q` | LLM label for every ad: 1 = negative (Attack or Contrast) |
| `ads.Q_tone` | the LLM's full answer: Promote / Contrast / Attack |
| `ads.X` | 1 = Facebook, 0 = TV |
| `ads.n_words` | length of the ad |
| `R` | 1 if the ad is in the expert sample |
| `PI` | the chance each ad had of being sampled |
| `expert_labels(R)` | expert label Y where `R == 1`, NaN elsewhere |
| `ols(y, x)` | returns `(slope, robust SE)` |

**Rule:** only use expert labels through `expert_labels(R)`. Using `ads.Y` directly is cheating: in a real project you don't have it.

## Part 1: the simplest DSL

DSL replaces each ad's outcome with a **corrected label**

$$\tilde Y_i = g_i + \frac{R_i}{\pi}\,(Y_i - g_i)$$

where $g_i$ is any guess of $Y_i$. Start with the LLM label itself: $g_i = Q_i$. Then regress $\tilde Y$ on $X$.

- For ads without an expert label, $R_i = 0$, so the correction term is 0. But `Y` is NaN there, and `0 * NaN` is NaN in Python. Fill those in before multiplying.
- Discuss: why is $\frac{1}{\pi}$ there? What is the average of $\tilde Y_i - Y_i$ over the random draw of $R$?

In [ ]:
def dsl_simple(R):
    Y = expert_labels(R)     # NaN where R == 0
    g = ads.Q                # our guess of Y: the LLM label

    # TODO: build the corrected label y_tilde, then return ols(y_tilde, ads.X)
    raise NotImplementedError

In [ ]:
b, se = dsl_simple(R)
print(f"DSL (g = Q)  {b:6.3f}  [{b - 1.96*se:6.3f}, {b + 1.96*se:6.3f}]")
print(f"Truth        {TRUTH:6.3f}")

## Part 2: a better guess $g$

Any $g$ keeps the estimate unbiased, but a better $g$ makes the interval narrower. Predict $Y$ from the LLM's full answer and the ad's features, using a logistic regression trained on the expert-labelled ads.

**Cross-fitting:** an ad's own expert label must not be used to predict its $g$. Split the ads into `k` folds; for each fold, train on the *expert-labelled ads in the other folds* and predict $g$ for the ads in this fold.

Hints: `KFold(k, shuffle=True, random_state=seed).split(features)` gives `(train, test)` index arrays; `LogisticRegression(max_iter=2000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]`.

In [ ]:
features = pd.get_dummies(ads.Q_tone).astype(float)   # one column per LLM answer
features["X"] = ads.X
features["len"] = np.log(ads.n_words)
features["QX"] = ads.Q * ads.X                        # lets the LLM's accuracy differ by platform

def dsl(R, k=5, seed=0):
    Y = expert_labels(R)
    g = np.zeros(N_CORPUS)

    # TODO 1: cross-fit g (see the hints above)
    # TODO 2: corrected label y_tilde, as in Part 1 but with this g
    # TODO 3: return ols(y_tilde, ads.X)
    raise NotImplementedError

In [ ]:
for name, f in [("LLM only", llm_only), ("Supervised", supervised),
                ("DSL (g = Q)", dsl_simple), ("DSL", dsl)]:
    b, se = f(R)
    print(f"{name:12s} {b:6.3f}  [{b - 1.96*se:6.3f}, {b + 1.96*se:6.3f}]")
print(f"{'Truth':12s} {TRUTH:6.3f}")

## Part 3: does it work every time?

One random sample can be lucky. Redraw the expert sample many times and check how often each method's 95% interval covers the truth (it should be about 95%).

In [ ]:
def simulate(methods):
    rows = []
    for r in range(REPEATS):
        R = draw(N_CORPUS, N_EXPERT, r)
        for name, f in methods:
            b, se = f(R)
            rows.append((name, b, se, abs(b - TRUTH) <= 1.96 * se))
    res = pd.DataFrame(rows, columns=["method", "estimate", "se", "covers_truth"])
    summary = res.groupby("method", sort=False).agg(mean_estimate=("estimate", "mean"),
                                                    coverage=("covers_truth", "mean"),
                                                    mean_se=("se", "mean")).round(3)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    for name, d in res.groupby("method", sort=False):
        ax[0].hist(d.estimate, bins=25, alpha=.6, label=name)
    ax[0].axvline(TRUTH, color="k", ls="--", label="Truth"); ax[0].set_title("Estimates"); ax[0].legend()
    summary.coverage.plot.bar(ax=ax[1], rot=0); ax[1].axhline(.95, color="k", ls="--")
    ax[1].set_ylim(0, 1); ax[1].set_title("How often the 95% interval covers the truth")
    plt.tight_layout()
    return summary

simulate([("LLM only", llm_only), ("Supervised", supervised),
          ("DSL (g = Q)", dsl_simple), ("DSL", dsl)])

## Discuss

1. Which methods cover the truth about 95% of the time? Why do the others fail?
2. Compare `mean_se` for the two DSL versions. What did the better $g$ buy you?
3. Set `N_EXPERT = 40` or `150` in the setup cell and rerun everything. What changes?
4. What would go wrong if the experts labelled the ads the LLM was *least sure* about, instead of a random sample?